In [1]:
import sys

# Add the desired path
sys.path.append('/Users/noahyoungs/VSCode/ai-presenter-v2/backend')

# Verify the path has been added
print(sys.path)



['/Users/noahyoungs/VSCode/ai-presenter-v2/backend/slides/images', '/Users/noahyoungs/VSCode/ai-presenter-v2/backend/.conda/lib/python311.zip', '/Users/noahyoungs/VSCode/ai-presenter-v2/backend/.conda/lib/python3.11', '/Users/noahyoungs/VSCode/ai-presenter-v2/backend/.conda/lib/python3.11/lib-dynload', '', '/Users/noahyoungs/.local/lib/python3.11/site-packages', '/Users/noahyoungs/VSCode/ai-presenter-v2/backend/.conda/lib/python3.11/site-packages', '/Users/noahyoungs/VSCode/Python/main_content_extractor', '/Users/noahyoungs/VSCode/ai-presenter-v2/backend']


In [2]:

from typing import List, Literal, Optional, Callable, Tuple, Union, Dict
from pydantic import BaseModel, Field
from pydantic_async_validation import AsyncValidationModelMixin, async_field_validator
from slides.xml_instructor import create_dynamic_enum_validator, generate_variable_input_xml, get_basemodel_instance_from_llm, parse_xml_to_model, generate_xml_from_instance
from portkey_ai import AsyncPortkey

from dotenv import load_dotenv
from slides.slide_models import Slide
from slides.images.generate_image import openai_styles_guidance, stability_styles_guidance, openai_style_presets, stability_style_presets

In [3]:
class DalleGenerationTemplate(BaseModel):
    reasoning: str
    prompt: str
    style: Literal["vivid", "natural", "default"]
    message_to_user: str

class StabilityGenerationTemplate(BaseModel):
    reasoning: str
    prompt: str
    style: Literal[
        "default", "3d-model", "analog-film", "anime", "cinematic", "comic-book",
        "fantasy-art", "isometric", "line-art", "low-poly", "modeling-compound",
        "neon-punk", "origami", "photographic", "pixel-art"
    ]
    negative_prompt: Optional[str] = None
    message_to_user: str
    
class OldDalleGenerationTemplate(BaseModel):
    prompt: str
    style: Literal["vivid", "natural", "default"]

class OldStabilityGenerationTemplate(BaseModel):
    prompt: str
    style: Literal[
        "default", "3d-model", "analog-film", "anime", "cinematic", "comic-book",
        "fantasy-art", "isometric", "line-art", "low-poly", "modeling-compound",
        "neon-punk", "origami", "photographic", "pixel-art"
    ]
    negative_prompt: Optional[str] = None

In [4]:
test_dalle = DalleGenerationTemplate.model_validate({
    "reasoning": "I want to generate an image of a cat",
    "prompt": "A realistic image of a cat",
    "style": "vivid",
    "message_to_user": "Here is your image of a cat"
})

In [5]:
test_stability = StabilityGenerationTemplate.model_validate({
    "reasoning": "I want to generate an image of a cat",
    "prompt": "A realistic image of a cat",
    "style": "default",
    "negative_prompt": "An unrealistic image of a cat",
    "message_to_user": "Here is your image of a cat"
})

test_old_stability_2 = OldStabilityGenerationTemplate.model_validate({
    "reasoning": "I want to generate an image of a cat",
    "prompt": "A realistic image of a cat",
    "style": "3d-model",
})

In [6]:
test_old_dalle = OldDalleGenerationTemplate.model_validate({
    "prompt": "A realistic image of a cat",
    "style": "vivid"
})

In [7]:
OldStabilityGenerationTemplate.schema()

{'properties': {'prompt': {'title': 'Prompt', 'type': 'string'},
  'style': {'enum': ['default',
    '3d-model',
    'analog-film',
    'anime',
    'cinematic',
    'comic-book',
    'fantasy-art',
    'isometric',
    'line-art',
    'low-poly',
    'modeling-compound',
    'neon-punk',
    'origami',
    'photographic',
    'pixel-art'],
   'title': 'Style',
   'type': 'string'},
  'negative_prompt': {'anyOf': [{'type': 'string'}, {'type': 'null'}],
   'default': None,
   'title': 'Negative Prompt'}},
 'required': ['prompt', 'style'],
 'title': 'OldStabilityGenerationTemplate',
 'type': 'object'}

In [8]:
xml = "<prompt>\nA realistic image of a cat\n</prompt>\n<style>\ndefault\n</style>\n<negative_prompt>\nAn unrealistic image of a cat\n</negative_prompt>"

model = parse_xml_to_model(xml, OldStabilityGenerationTemplate)

Whole Schema: {'properties': {'prompt': {'title': 'Prompt', 'type': 'string'}, 'style': {'enum': ['default', '3d-model', 'analog-film', 'anime', 'cinematic', 'comic-book', 'fantasy-art', 'isometric', 'line-art', 'low-poly', 'modeling-compound', 'neon-punk', 'origami', 'photographic', 'pixel-art'], 'title': 'Style', 'type': 'string'}, 'negative_prompt': {'anyOf': [{'type': 'string'}, {'type': 'null'}], 'default': None, 'title': 'Negative Prompt'}}, 'required': ['prompt', 'style'], 'title': 'OldStabilityGenerationTemplate', 'type': 'object'}
Element: prompt
Schema: {'title': 'Prompt', 'type': 'string'}
Element: prompt
Element: style
Schema: {'enum': ['default', '3d-model', 'analog-film', 'anime', 'cinematic', 'comic-book', 'fantasy-art', 'isometric', 'line-art', 'low-poly', 'modeling-compound', 'neon-punk', 'origami', 'photographic', 'pixel-art'], 'title': 'Style', 'type': 'string'}
Element: style
Element: negative_prompt
Schema: {'anyOf': [{'type': 'string'}, {'type': 'null'}], 'default

In [9]:
model.dict()

{'prompt': 'A realistic image of a cat',
 'style': 'default',
 'negative_prompt': 'An unrealistic image of a cat'}

In [10]:
generate_xml_from_instance(test_old_stability_2)

'<prompt>\nA realistic image of a cat\n</prompt>\n<style>\n3d-model\n</style>'

In [11]:
xml_2 = "<prompt>\nA realistic image of a cat\n</prompt>\n<style>\n3d-model\n</style>"

model_2 = parse_xml_to_model(xml_2, OldStabilityGenerationTemplate)

Whole Schema: {'properties': {'prompt': {'title': 'Prompt', 'type': 'string'}, 'style': {'enum': ['default', '3d-model', 'analog-film', 'anime', 'cinematic', 'comic-book', 'fantasy-art', 'isometric', 'line-art', 'low-poly', 'modeling-compound', 'neon-punk', 'origami', 'photographic', 'pixel-art'], 'title': 'Style', 'type': 'string'}, 'negative_prompt': {'anyOf': [{'type': 'string'}, {'type': 'null'}], 'default': None, 'title': 'Negative Prompt'}}, 'required': ['prompt', 'style'], 'title': 'OldStabilityGenerationTemplate', 'type': 'object'}
Element: prompt
Schema: {'title': 'Prompt', 'type': 'string'}
Element: prompt
Element: style
Schema: {'enum': ['default', '3d-model', 'analog-film', 'anime', 'cinematic', 'comic-book', 'fantasy-art', 'isometric', 'line-art', 'low-poly', 'modeling-compound', 'neon-punk', 'origami', 'photographic', 'pixel-art'], 'title': 'Style', 'type': 'string'}
Element: style


In [12]:
model_2.dict()

{'prompt': 'A realistic image of a cat',
 'style': '3d-model',
 'negative_prompt': None}

In [11]:
class BulletPoint(BaseModel):
        content: str = Field(..., min_length=1, max_length=80)
        excerpt_id: Optional[str] = Field(None, min_length=8, max_length=8)

        validate_excerpt_id = create_dynamic_enum_validator(
            'valid_ids', 'excerpt_id')

class ShortBulletsSlide(BaseModel):
    slide_title: Optional[str] = Field(None, min_length=1, max_length=75)
    bullet_points: Optional[List[BulletPoint]
                            ] = Field(None, min_items=2, max_items=5)
    sequential_order: Optional[Literal["yes", "no"]] = None
    message_to_user: str

class ShortBulletsSlideRequired(BaseModel):
    slide_title: str = Field(..., min_length=1, max_length=75)
    bullet_points: List[BulletPoint] = Field(..., min_items=2, max_items=5)
    sequential_order: Literal["yes", "no"]
    message_to_user: str

xml = """\
<sequential_order>
no
</sequential_order>
<message_to_user>
I have removed the bullet point related to earning miles through financial partners, as per your request. The slide now has 2 bullet points, which is within the required 2-5 bullet point range. The bullet points are also within the 80 character limit and do not contain any nested content.
Show less
</message_to_user>"""

model = parse_xml_to_model(xml, ShortBulletsSlide)

Whole Schema: {'$defs': {'BulletPoint': {'properties': {'content': {'maxLength': 80, 'minLength': 1, 'title': 'Content', 'type': 'string'}, 'excerpt_id': {'anyOf': [{'maxLength': 8, 'minLength': 8, 'type': 'string'}, {'type': 'null'}], 'default': None, 'title': 'Excerpt Id'}}, 'required': ['content'], 'title': 'BulletPoint', 'type': 'object'}}, 'properties': {'slide_title': {'anyOf': [{'maxLength': 75, 'minLength': 1, 'type': 'string'}, {'type': 'null'}], 'default': None, 'title': 'Slide Title'}, 'bullet_points': {'anyOf': [{'items': {'$ref': '#/$defs/BulletPoint'}, 'maxItems': 5, 'minItems': 2, 'type': 'array'}, {'type': 'null'}], 'default': None, 'title': 'Bullet Points'}, 'sequential_order': {'anyOf': [{'enum': ['yes', 'no'], 'type': 'string'}, {'type': 'null'}], 'default': None, 'title': 'Sequential Order'}, 'message_to_user': {'title': 'Message To User', 'type': 'string'}}, 'required': ['message_to_user'], 'title': 'ShortBulletsSlide', 'type': 'object'}
Element: sequential_order
S

In [12]:
model.dict()

{'slide_title': None,
 'bullet_points': None,
 'sequential_order': 'no',
 'message_to_user': 'I have removed the bullet point related to earning miles through financial partners, as per your request. The slide now has 2 bullet points, which is within the required 2-5 bullet point range. The bullet points are also within the 80 character limit and do not contain any nested content.\nShow less'}